# Phase 3 - The Three Quantum Encodings, Explained

This notebook does **no training**. Its only job is to make the three
encodings concrete: for one sample data point, we build each encoding circuit
and look at the quantum state it produces. Understanding what each encoding
*does to a single point* is the key to interpreting the classifier results
later.

The three encodings are:

- **Angle encoding** - one rotation per feature, no entanglement.
- **Amplitude encoding** - features stored directly in state amplitudes.
- **ZZ feature map** - rotations plus entangling gates.

In [ ]:
"""03_quantum_encodings_explained.ipynb"""

# Cell 01 - Imports

import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import z_feature_map, zz_feature_map
from qiskit.quantum_info import Statevector
from qiskit_machine_learning.circuit.library import raw_feature_vector

import qml_utils as hu
from qml_utils import as_latex


---
**Cell 02 - Pick one sample point.**
We take a single two-feature point and use it throughout the notebook so the
three encodings can be compared directly. The values are chosen inside
$[0, \pi]$, the same range produced by the scaler in Phase 2.

In [ ]:
# Cell 02 - One example data point used for all three encodings

sample = np.array([0.9, 2.4])
print(f"Sample point (feature 1, feature 2): {sample}")

---
**Cell 03 - Angle encoding.**
Angle encoding turns each feature into a rotation angle on its own qubit.
Below we build it two ways:

- **By hand** with $R_y$ rotations, so you can see exactly what "angle
  encoding" means: feature 1 rotates qubit 0, feature 2 rotates qubit 1.
- **With Qiskit's `z_feature_map`**, the library's first-order Pauli-Z map
  (a Hadamard followed by a phase rotation per qubit). This is the version we
  use in the VQC in Phase 4.

The crucial property: there are **no entangling gates**. Each qubit carries
one feature independently, so the qubits never become correlated.

In [ ]:
# Cell 03 - Angle encoding, by hand and via z_feature_map

# (a) Manual Ry angle encoding: one rotation per feature
manual = QuantumCircuit(2)
manual.ry(sample[0], 0)
manual.ry(sample[1], 1)
print("Manual Ry angle-encoding circuit:")
display(manual.draw(output="mpl"))
display(as_latex(Statevector(manual).data, prefix=r"\mathbf{Manual\;state}="))

# (b) Qiskit's built-in first-order Z feature map (no entanglement)
angle_map = z_feature_map(feature_dimension=2, reps=1)
bound_angle = angle_map.assign_parameters(sample)
print("z_feature_map angle-encoding circuit:")
display(bound_angle.draw(output="mpl"))
display(
    as_latex(Statevector(bound_angle).data, prefix=r"\mathbf{z\_feature\_map\;state}=")
)

---
**Cell 04 - Amplitude encoding.**
Amplitude encoding writes the feature vector directly into the amplitudes of
the quantum state. Two consequences:

- The vector length must be a power of two, so we pad our 2 features up to 4
  values (giving $\log_2 4 = 2$ qubits).
- A quantum state must have unit length, so the padded vector is normalized.

Normalization throws away the overall magnitude and keeps only the direction.
That is a real limitation worth remembering.

Unlike the other encodings, amplitude encoding has no fixed gate pattern.
Qiskit *synthesizes* a state-preparation circuit from the amplitudes using an
`initialize` instruction. Calling `.decompose()` only reveals the synthesis
algorithm's internal `isometry_to_uncompute` box, not real gates, so to see
what actually runs we **transpile** to a concrete gate set. The transpiled
circuit also contains `reset` gates, because `initialize` is defined as
"reset to $|0
angle$, then prepare the state."

In [ ]:
# Cell 04 - Amplitude encoding via raw_feature_vector

# Pad [f1, f2] -> [f1, f2, 0, 0] and normalize to unit length
prepared = hu.pad_and_normalize(sample.reshape(1, -1), num_amplitudes=4)[0]
print(f"Padded and normalized amplitudes: {prepared}")
print(f"Norm (should be 1.0): {np.linalg.norm(prepared):.6f}")

amp_map = raw_feature_vector(feature_dimension=4)
bound_amp = amp_map.assign_parameters(prepared)

# .decompose() only exposes the opaque synthesis instruction
print("High-level view (decompose) - the opaque isometry instruction:")
display(bound_amp.decompose().draw(output="mpl"))

# Transpiling synthesizes the real, executable gates
runnable = transpile(bound_amp, basis_gates=["u", "cx"], optimization_level=3)
print(f"Executable view (transpiled) - ops {dict(runnable.count_ops())}:")
display(runnable.draw(output="mpl"))

display(as_latex(Statevector(bound_amp).data, prefix=r"\mathbf{Amplitude\;state}="))

---
**Cell 04b (optional) - A richer amplitude-encoding circuit.**
The circuit above looks sparse for two reasons: zero-padding left two
amplitudes at zero, and *any* 2-qubit state can be prepared with at most a
single entangling (CX) gate. Two changes make the synthesized circuit fuller:

1. **Use all the amplitudes.** Instead of padding with zeros, pad with a
   feature product so all four amplitudes carry information. This is itself a
   legitimate (richer) encoding choice: it injects the nonlinear feature
   $f_1 f_2$ into the state.
2. **Use more qubits.** State-preparation cost grows quickly with qubit count.
   An 8-amplitude vector needs $\log_2 8 = 3$ qubits and many more gates.

This cell is illustrative only. The project itself stays with the simple
4-amplitude, 2-qubit setup used in Phase 5.

In [ ]:
# Cell 04b - Richer amplitude encodings (illustration only)

# (a) Product padding: [f1, f2, f1*f2, 1] so all four amplitudes are nonzero
f1, f2 = sample
product_vector = np.array([f1, f2, f1 * f2, 1.0])
product_vector = product_vector / np.linalg.norm(product_vector)
print(f"Product-padded amplitudes: {np.round(product_vector, 3)}")

amp_product = raw_feature_vector(feature_dimension=4).assign_parameters(product_vector)
runnable_product = transpile(amp_product, basis_gates=["u", "cx"], optimization_level=3)
print(
    f"4 amplitudes / 2 qubits -> depth {runnable_product.depth()}, ops {dict(runnable_product.count_ops())}"
)
display(runnable_product.draw(output="mpl"))

# (b) Eight amplitudes on 3 qubits: state preparation grows with qubit count
rng = np.random.default_rng(1)
big_vector = rng.random(8)
big_vector = big_vector / np.linalg.norm(big_vector)

amp_big = raw_feature_vector(feature_dimension=8).assign_parameters(big_vector)
runnable_big = transpile(amp_big, basis_gates=["u", "cx"], optimization_level=3)
print(
    f"8 amplitudes / 3 qubits -> depth {runnable_big.depth()}, ops {dict(runnable_big.count_ops())}"
)
display(runnable_big.draw(output="mpl"))

---
**Cell 05 - ZZ feature map.**
The ZZ feature map starts like angle encoding (a rotation per feature) but
then adds **entangling gates** that act on pairs of qubits using the product
of their feature values. Those entangling blocks are what let the encoding
represent feature *interactions* that single-qubit rotations cannot.

Look for the two-qubit gates (CX gates) in the drawing - they are absent from
angle encoding. The hypothesis of this project is that these entangling gates
will produce a measurably different decision boundary.

In [ ]:
# Cell 05 - ZZ feature map (entangling)

zz_map = zz_feature_map(feature_dimension=2, reps=1)
bound_zz = zz_map.assign_parameters(sample)
print("zz_feature_map circuit (decomposed to show entangling gates):")
display(bound_zz.decompose().draw(output="mpl"))
display(as_latex(Statevector(bound_zz).data, prefix=r"\mathbf{ZZ\;state}="))

---
**Cell 06 - Compare the three encodings.**
Finally we summarize the structural differences: how many qubits each uses
and how deep each circuit is. Circuit depth is a rough proxy for how much
work the encoding asks of the hardware. We transpile to a common gate set so
the depths are comparable.

In [ ]:
# Cell 06 - Side-by-side structural comparison

basis = ["rx", "ry", "rz", "cx"]


def describe(circuit, label):
    flat = transpile(circuit, basis_gates=basis, optimization_level=0)
    ops = dict(flat.count_ops())
    n_cx = ops.get("cx", 0)
    print(
        f"{label:<22} qubits={circuit.num_qubits}  depth={flat.depth():<3}  cx_gates={n_cx}"
    )


print(f"{'Encoding':<22} structure")
print("-" * 60)
describe(bound_angle, "Angle (z_feature_map)")
describe(bound_amp, "Amplitude")
describe(bound_zz, "ZZ feature map")
print()
print("Angle encoding has no CX gates (no entanglement).")
print("The ZZ feature map adds CX gates that entangle the qubits.")

---
## What to take away

- **Angle encoding** maps features to independent rotations. Simple, shallow,
  no entanglement.
- **Amplitude encoding** packs features into amplitudes. Compact in qubits
  but discards magnitude and needs a more involved state preparation.
- **ZZ feature map** adds entangling gates that capture feature interactions.

In the next three notebooks we attach a trainable circuit (an *ansatz*) to
each encoding and see how well the resulting classifier separates the moons.